f(x0, x1, x2) = 3.x0 + 2.x1 + x2 - 4.x0.x1 - 2.x1.x2

| x0.x1.x2 | QUBO value |
| :--- | :--- |
| 000 | 0 |
| 001 | 1 |
| 010 | 2 |
| 011 | 1 |
| 100 | 3 |
| 101 | 4 |
| 110 | 1 |
| 111 | 0 |

In [1]:
from qiskit_optimization import QuadraticProgram

qp = QuadraticProgram()

# Define binary variables
qp.binary_var("x0")
qp.binary_var("x1")
qp.binary_var("x2")

# Define QUBO objective
qp.minimize(
    linear={
        "x0": 3,
        "x1": 2,
        "x2": 1
    },
    quadratic={
        ("x0", "x1"): -4,
        ("x1", "x2"): -2
    }
)

print(qp.prettyprint())

Problem name: 

Minimize
  -4*x0*x1 - 2*x1*x2 + 3*x0 + 2*x1 + x2

Subject to
  No constraints

  Binary variables (3)
    x0 x1 x2



## <u>**Ising Hamiltonian**</u>

Convert binary variables into spin variables.

x_i = (1 - Z_i) / 2

where Z_i is the Pauli-Z operator.

For example:

x_0.x_1 = (1 - Z_0)(1 - Z_1) / 4

x_0.x_1= (1 - Z_0 - Z_1 + Z_0.Z_1) / 4

The result is an Ising Hamiltonian:
H_C = c.I + sum_i h_i.Z_i + sum_(i<j) J_(ij).Z_i.Z_j

Here:
- I is the identity operator.
- Z_i represents an individual qubit.
- Z_i.Z_j represents interaction between two qubits.
- c is a constant offset.

In [2]:
from qiskit_optimization.converters import QuadraticProgramToQubo

converter = QuadraticProgramToQubo()
qubo = converter.convert(qp)

operator, offset = qubo.to_ising()

print("Ising Hamiltonian:")
print(operator)

print("Offset:", offset)

Ising Hamiltonian:
SparsePauliOp(['IIZ', 'IZI', 'IZZ', 'ZZI'],
              coeffs=[-0.5+0.j,  0.5+0.j, -1. +0.j, -0.5+0.j])
Offset: 1.5


In [3]:
from qiskit.quantum_info import SparsePauliOp

# Example Ising Hamiltonian
H = SparsePauliOp.from_list([
    ("III", 0.5),
    ("IIZ", -0.5),
    ("IZI", 0.5),
    ("ZII", 0.5),
    ("IZZ", -1.0),
    ("ZZI", -0.5)
])

print(H)

SparsePauliOp(['III', 'IIZ', 'IZI', 'ZII', 'IZZ', 'ZZI'],
              coeffs=[ 0.5+0.j, -0.5+0.j,  0.5+0.j,  0.5+0.j, -1. +0.j, -0.5+0.j])
